# Stock Price Prediction Using Deep Learning: LSTM and GRU Models
### Subtitle: A Deep Learning-Based Time Series Analysis of ADANIPORTS Stock Prices
**Academic Context:** MBA Data Science & AI Academic Project, Demonstration, and Viva Voce

---
## 1. Project Objective
This notebook demonstrates an end-to-end deep learning time-series regression workflow on historical quotes of **ADANIPORTS** (2007–2021). We formulate the task as predicting the **next trading day's closing price** ($Close_{t+1}$) using a sliding window of the previous **60 trading days** ($t-59$ to $t$).

Key architectural pillars:
1. **Zero Data Leakage:** Strict chronological 80/20 train/test split. Scaler fitted exclusively on the training interval.
2. **Architectural Comparison:** LSTM (Long Short-Term Memory) vs. GRU (Gated Recurrent Unit).
3. **Econometric Diagnostics:** MAE, MSE, RMSE, MAPE, and R2 Score evaluated on unscaled INR (Rs.) prices.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print(f'TensorFlow Version: {tf.__version__}')

## 2. Dataset Ingestion & Auto-Detection of Columns
We load `ADANIPORTS-selected-columns-2.csv` dynamically without hardcoding assumptions about column ordering.

In [2]:
DATA_PATH = os.path.join('..', 'data', 'ADANIPORTS-selected-columns-2.csv')
df = pd.read_csv(DATA_PATH)
print('Initial Dataset Shape:', df.shape)
print('Detected Columns:', df.columns.tolist())
df.head(5)

## 3. Data Preprocessing & Validation
Steps executed:
- Convert Date column to datetime format.
- Chronological sorting in ascending order.
- Checking and handling missing values (forward fill + backward fill).
- Validating price integrity ($Close > 0$ and $High >= Low$).

In [3]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date']).sort_values(by='Date', ascending=True).reset_index(drop=True)

numeric_cols = ['Prev Close', 'Open', 'High', 'Low', 'Last', 'Close', 'VWAP']
existing_numeric = [c for c in numeric_cols if c in df.columns]
for col in existing_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df[existing_numeric] = df[existing_numeric].ffill().bfill()

print(f'Final Usable Records: {len(df)}')
print(f"Date Range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")

## 4. Exploratory Data Analysis & Technical Features
Compute backward-looking indicators:
- Daily Return (%)
- 20-day Simple Moving Average (SMA 20)
- 50-day Simple Moving Average (SMA 50)
- 100-day Simple Moving Average (SMA 100)
- 20-day Rolling Volatility

In [4]:
df['Daily_Return'] = df['Close'].pct_change() * 100.0
df['Daily_Return'] = df['Daily_Return'].fillna(0.0)
df['SMA_20'] = df['Close'].rolling(window=20, min_periods=1).mean()
df['SMA_50'] = df['Close'].rolling(window=50, min_periods=1).mean()
df['SMA_100'] = df['Close'].rolling(window=100, min_periods=1).mean()
df['Volatility_20'] = df['Daily_Return'].rolling(window=20, min_periods=1).std().fillna(0.0)

plt.figure(figsize=(14, 6))
plt.plot(df['Date'], df['Close'], label='ADANIPORTS Close (Rs.)', color='#0F172A', lw=1.2)
plt.plot(df['Date'], df['SMA_20'], label='SMA 20', color='#2563EB', lw=1.5)
plt.plot(df['Date'], df['SMA_50'], label='SMA 50', color='#F59E0B', lw=1.5)
plt.plot(df['Date'], df['SMA_100'], label='SMA 100', color='#EF4444', lw=1.8)
plt.title('ADANIPORTS Historical Close & Moving Averages', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (Rs. INR)')
plt.legend()
plt.show()

## 5. Chronological Split (80/20) & Isolated MinMax Scaling
Strictly fit scaler on training data only.

In [5]:
train_size = int(len(df) * 0.80)
train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_df[['Close']].values)

sequence_length = 60
test_context = df.iloc[train_size - sequence_length:][['Close']].values
test_scaled = scaler.transform(test_context)

print(f'Train instances: {len(train_df)}, Test instances: {len(test_df)}')

## 6. Sequence Creation (60 Trading Days Sliding Window)

In [6]:
def make_sequences(arr, seq_len=60):
    X, y = [], []
    for i in range(seq_len, len(arr)):
        X.append(arr[i-seq_len:i])
        y.append(arr[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = make_sequences(train_scaled, sequence_length)
X_test, y_test = make_sequences(test_scaled, sequence_length)
y_test_unscaled = test_df['Close'].values

print('X_train shape:', X_train.shape, 'y_train shape:', y_train.shape)
print('X_test shape:', X_test.shape, 'y_test shape:', y_test.shape)

## 7. Model Architectures & Training
- LSTM: Stacked (64, 32) + Dropout(0.2) + Dense(16, relu) + Dense(1)
- GRU: Stacked (64, 32) + Dropout(0.2) + Dense(16, relu) + Dense(1)

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def create_model(rnn_type='LSTM'):
    Layer = LSTM if rnn_type == 'LSTM' else GRU
    model = Sequential([
        Input(shape=(sequence_length, 1)),
        Layer(64, return_sequences=True),
        Dropout(0.2),
        Layer(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ], name=f'{rnn_type}_Predictor')
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

lstm_model = create_model('LSTM')
gru_model = create_model('GRU')

callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5)
]

print('Training LSTM...')
lstm_model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.1, shuffle=False, callbacks=callbacks, verbose=1)

print('Training GRU...')
gru_model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.1, shuffle=False, callbacks=callbacks, verbose=1)

## 8. Evaluation & Comparison

In [8]:
lstm_preds = scaler.inverse_transform(lstm_model.predict(X_test)).flatten()
gru_preds = scaler.inverse_transform(gru_model.predict(X_test)).flatten()

def evaluate(y_true, y_p):
    mae = mean_absolute_error(y_true, y_p)
    mse = mean_squared_error(y_true, y_p)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_p) / y_true)) * 100.0
    r2 = r2_score(y_true, y_p)
    return {'MAE (Rs.)': mae, 'MSE': mse, 'RMSE (Rs.)': rmse, 'MAPE (%)': mape, 'R2 Score': r2}

benchmark = pd.DataFrame({
    'LSTM': evaluate(y_test_unscaled, lstm_preds),
    'GRU': evaluate(y_test_unscaled, gru_preds)
}).T

print(benchmark)

## 9. Visualizing Trajectories

In [9]:
plt.figure(figsize=(14, 6))
plt.plot(test_df['Date'], y_test_unscaled, label='Actual Close', color='#0F172A', lw=2)
plt.plot(test_df['Date'], lstm_preds, label='LSTM Predicted', color='#2563EB', lw=1.8, linestyle='--')
plt.plot(test_df['Date'], gru_preds, label='GRU Predicted', color='#10B981', lw=1.8, linestyle=':')
plt.title('ADANIPORTS Test Set: Actual vs LSTM vs GRU Predictions', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (Rs. INR)')
plt.legend()
plt.show()

## 10. Next-Day Inference

In [10]:
latest_window = df.tail(60)[['Close']].values
latest_scaled = scaler.transform(latest_window).reshape(1, 60, 1)
forecast_scaled = gru_model.predict(latest_scaled)
forecast_close = scaler.inverse_transform(forecast_scaled)[0, 0]
latest_val = float(df['Close'].iloc[-1])

print(f'Latest Observed Close ({df["Date"].iloc[-1].strftime("%Y-%m-%d")}): Rs. {latest_val:.2f}')
print(f'Predicted Next Trading Day Close: Rs. {forecast_close:.2f}')
print(f'Expected Delta: Rs. {forecast_close - latest_val:+.2f} ({(forecast_close - latest_val)/latest_val * 100:+.2f}%)')